In [16]:
!pip install -q rdkit meeko biopython plotly pandas scipy

In [2]:
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina
!chmod +x vina
!./vina --version

AutoDock Vina v1.2.5


In [3]:
!pip install -q gemmi

In [4]:
import rdkit, Bio, meeko, gemmi
print("RDKit:", rdkit.__version__)
print("Meeko and gemmi imported fine")

RDKit: 2026.03.6
Meeko and gemmi imported fine


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
!wget -q https://files.rcsb.org/download/6LU7.pdb -O 6LU7.pdb

from Bio.PDB import PDBParser
structure = PDBParser(QUIET=True).get_structure("6LU7", "6LU7.pdb")

for chain in structure[0]:
    names = [res.get_resname() for res in chain]
    print("Chain", chain.id, "|", len(names), "residues |", sorted(set(names))[:10])

Chain A | 390 residues | ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'HOH']
Chain C | 6 residues | ['010', '02J', 'ALA', 'LEU', 'PJE', 'VAL']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
from Bio.PDB import PDBIO, Select

class ProteinOnly(Select):
    def accept_chain(self, chain):
        return chain.id == "A"
    def accept_residue(self, residue):
        return residue.id[0] == " "   # keep only normal amino acids (drops water)

class LigandOnly(Select):
    def accept_chain(self, chain):
        return chain.id == "C"

io = PDBIO()
io.set_structure(structure)
io.save("protein.pdb", ProteinOnly())
io.save("N3_ligand.pdb", LigandOnly())

!grep -c -E "^(ATOM|HETATM)" protein.pdb
!grep -c HOH protein.pdb
!grep -c -E "^(ATOM|HETATM)" N3_ligand.pdb

2367
0
49


In [7]:
!apt-get -qq install -y openbabel > /dev/null
!obabel protein.pdb -O protein.pdbqt -xr -p 7.4
!obabel N3_ligand.pdb -O N3.pdbqt -p 7.4
!grep -c -E "^(ATOM|HETATM)" protein.pdbqt
!grep -c -E "^(ATOM|HETATM)" N3.pdbqt
!grep TORSDOF N3.pdbqt

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is protein.pdb)

1 molecule converted
1 molecule converted
2888
54
TORSDOF 18


In [ ]:
import numpy as np
coords = np.array([a.coord for a in structure[0]["C"].get_atoms()])
center = coords.mean(axis=0)
size = coords.max(axis=0) - coords.min(axis=0) + 10
print("center:", center.round(1))
print("size:", size.round(1))

In [9]:
cx, cy, cz = [round(float(v), 2) for v in center]
sx, sy, sz = [round(float(v), 2) for v in size]

!./vina --receptor protein.pdbqt --ligand N3.pdbqt \
  --center_x {cx} --center_y {cy} --center_z {cz} \
  --size_x {sx} --size_y {sy} --size_z {sz} \
  --exhaustiveness 16 --num_modes 5 --out N3_docked.pdbqt

AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

In [ ]:
def heavy_atoms(lines):
    atoms = {}
    for l in lines:
        if l.startswith(("ATOM", "HETATM")):
            if l[77:79].strip() not in ("H", "HD"):
                atoms[int(l[6:11])] = np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
    return atoms

ref = heavy_atoms(open("N3.pdbqt").read().splitlines())

models = open("N3_docked.pdbqt").read().split("ENDMDL")
for i, m in enumerate(models[:-1], 1):
    pose = heavy_atoms(m.splitlines())
    diffs = [ref[k] - pose[k] for k in ref if k in pose]
    rmsd = np.sqrt(np.mean(np.sum(np.square(diffs), axis=1)))
    print(f"Pose {i}: RMSD = {rmsd:.2f} Å")

In [ ]:
!wget -q https://files.rcsb.org/download/6W63.pdb -O 6W63.pdb
s2 = PDBParser(QUIET=True).get_structure("6W63", "6W63.pdb")

class Prot(Select):
    def accept_chain(self, chain): return chain.id == "A"
    def accept_residue(self, res): return res.id[0] == " "

class Lig(Select):
    def accept_chain(self, chain): return chain.id == "A"
    def accept_residue(self, res): return res.get_resname() == "X77"

io = PDBIO(); io.set_structure(s2)
io.save("protein2.pdb", Prot())
io.save("X77.pdb", Lig())

!grep -c -E "^(ATOM|HETATM)" X77.pdb
!obabel protein2.pdb -O protein2.pdbqt -xr -p 7.4
!obabel X77.pdb -O X77.pdbqt -p 7.4
!grep TORSDOF X77.pdbqt

coords2 = np.array([a.coord for a in s2[0]["A"].get_atoms() if a.get_parent().get_resname() == "X77"])
center2 = coords2.mean(axis=0)
size2 = coords2.max(axis=0) - coords2.min(axis=0) + 10
print("center:", center2.round(1), "size:", size2.round(1))

In [ ]:
cx, cy, cz = [round(float(v), 2) for v in center2]
sx, sy, sz = [round(float(v), 2) for v in size2]

!./vina --receptor protein2.pdbqt --ligand X77.pdbqt \
  --center_x {cx} --center_y {cy} --center_z {cz} \
  --size_x {sx} --size_y {sy} --size_z {sz} \
  --exhaustiveness 16 --num_modes 5 --out X77_docked.pdbqt | tail -8

ref = heavy_atoms(open("X77.pdbqt").read().splitlines())
models = open("X77_docked.pdbqt").read().split("ENDMDL")
for i, m in enumerate(models[:-1], 1):
    pose = heavy_atoms(m.splitlines())
    diffs = [ref[k] - pose[k] for k in ref if k in pose]
    print(f"Pose {i}: RMSD = {np.sqrt(np.mean(np.sum(np.square(diffs), axis=1))):.2f} Å")

In [ ]:
import urllib.request, urllib.parse, json, os
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from meeko import MoleculePreparation, PDBQTWriterLegacy

names = ["nirmatrelvir", "ensitrelvir", "boceprevir", "darunavir", "atazanavir",
         "saquinavir", "indinavir", "nelfinavir", "amprenavir", "lopinavir", "ritonavir",
         "ebselen", "disulfiram", "carmofur", "baicalein", "quercetin", "luteolin",
         "myricetin", "curcumin", "favipiravir", "chloroquine", "hydroxychloroquine",
         "caffeine", "aspirin", "paracetamol", "ibuprofen"]

os.makedirs("ligands", exist_ok=True)

def get_smiles(name):
    url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
           + urllib.parse.quote(name) + "/property/SMILES/JSON")
    data = json.load(urllib.request.urlopen(url, timeout=30))
    p = data["PropertyTable"]["Properties"][0]
    return p.get("SMILES") or p.get("IsomericSMILES") or p.get("CanonicalSMILES")

library = []
for name in names:
    try:
        mol = Chem.AddHs(Chem.MolFromSmiles(get_smiles(name)))
        AllChem.EmbedMolecule(mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(mol)
        setup = MoleculePreparation().prepare(mol)[0]
        pdbqt, ok, err = PDBQTWriterLegacy.write_string(setup)
        if not ok:
            raise ValueError(err)
        open(f"ligands/{name}.pdbqt", "w").write(pdbqt)
        rot = rdMolDescriptors.CalcNumRotatableBonds(mol)
        mw = Descriptors.MolWt(mol)
        library.append((name, round(mw, 1), rot))
        print(f"OK   {name:20s} weight={mw:6.1f}  rotatable bonds={rot}")
    except Exception as e:
        print(f"FAIL {name}: {str(e)[:80]}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, json

W = "/content/drive/MyDrive/docking_project"
os.makedirs(W + "/docked", exist_ok=True)
shutil.copy("protein2.pdbqt", W)
shutil.copytree("ligands", W + "/ligands", dirs_exist_ok=True)
json.dump({"center": [float(v) for v in center2],
           "size": [float(v) for v in size2],
           "library": library}, open(W + "/setup.json", "w"))
print(os.listdir(W))

In [24]:
import os, json, subprocess, re, time
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

W = "/content/drive/MyDrive/docking_project"
if not os.path.exists("vina"):
    subprocess.run("wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina && chmod +x vina", shell=True)

setup = json.load(open(W + "/setup.json"))
cx, cy, cz = [round(v, 2) for v in setup["center"]]
sx, sy, sz = [round(v, 2) for v in setup["size"]]
BATCH = 5

todo = [m for m in setup["library"] if not os.path.exists(f"{W}/docked/{m[0]}_out.pdbqt")]
print(f"{len(setup['library']) - len(todo)} done, {len(todo)} remaining")

for name, mw, rot in todo[:BATCH]:
    t = time.time()
    subprocess.run(["./vina", "--receptor", f"{W}/protein2.pdbqt",
                    "--ligand", f"{W}/ligands/{name}.pdbqt",
                    "--center_x", str(cx), "--center_y", str(cy), "--center_z", str(cz),
                    "--size_x", str(sx), "--size_y", str(sy), "--size_z", str(sz),
                    "--exhaustiveness", "16", "--num_modes", "1",
                    "--out", f"{W}/docked/{name}_out.pdbqt"], capture_output=True)
    print(f"{name:20s} finished in {(time.time()-t)/60:.1f} min")
print("Batch complete. Run this cell again for the next batch.")

25 done, 0 remaining
Batch complete. Run this cell again for the next batch.


In [25]:
import os, re, json
import pandas as pd
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

W = "/content/drive/MyDrive/docking_project"
setup = json.load(open(W + "/setup.json"))

controls = ["caffeine", "aspirin", "paracetamol", "ibuprofen"]
natural = ["baicalein", "quercetin", "luteolin", "myricetin", "curcumin"]

rows = []
for name, mw, rot in setup["library"]:
    txt = open(f"{W}/docked/{name}_out.pdbqt").read()
    score = float(re.search(r"REMARK VINA RESULT:\s+(-?\d+\.\d+)", txt).group(1))
    best_pose = txt.split("ENDMDL")[0].splitlines()
    heavy = sum(1 for l in best_pose
                if l.startswith(("ATOM", "HETATM")) and l[77:79].strip() not in ("H", "HD"))
    group = ("negative control" if name in controls
             else "natural compound" if name in natural else "drug")
    rows.append({"compound": name, "group": group, "weight": mw,
                 "rotatable_bonds": rot, "heavy_atoms": heavy, "score": score,
                 "efficiency": round(-score / heavy, 3)})

df = pd.DataFrame(rows).sort_values("score").reset_index(drop=True)
df.to_csv(W + "/docking_results.csv", index=False)
print(df.to_string())
print()
print(df.groupby("group")[["score", "efficiency"]].mean().round(2))

              compound             group  weight  rotatable_bonds  heavy_atoms  score  efficiency
0          ensitrelvir              drug   531.9                5           37 -8.722       0.236
1           saquinavir              drug   670.9               13           49 -8.611       0.176
2           nelfinavir              drug   567.8               11           40 -8.580       0.214
3            lopinavir              drug   628.8               16           46 -8.275       0.180
4            indinavir              drug   613.8               13           45 -8.194       0.182
5            darunavir              drug   547.7               13           38 -7.977       0.210
6           boceprevir              drug   519.7                8           37 -7.915       0.214
7             luteolin  natural compound   286.2                5           21 -7.807       0.372
8            myricetin  natural compound   318.2                7           23 -7.795       0.339
9           atazanav

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [26]:
import plotly.express as px
from scipy.stats import spearmanr

hiv = ["darunavir", "atazanavir", "saquinavir", "indinavir", "nelfinavir",
       "amprenavir", "lopinavir", "ritonavir"]
covalent = ["nirmatrelvir", "boceprevir", "carmofur", "disulfiram"]
other = ["chloroquine", "hydroxychloroquine", "favipiravir"]

def regroup(row):
    n = row["compound"]
    if n == "ensitrelvir": return "non-covalent Mpro inhibitor"
    if n in covalent: return "covalent / cysteine-reactive"
    if n in hiv: return "HIV protease inhibitor"
    if n in other: return "other repurposed drug"
    return row["group"]

df["group2"] = df.apply(regroup, axis=1)
print(df.groupby("group2")["score"].agg(["count", "mean"]).round(2))

rho, p = spearmanr(df["heavy_atoms"], df["score"])
print(f"\nSize vs score: rho = {rho:.2f}, p = {p:.4f}")

fig = px.scatter(df, x="heavy_atoms", y="score", color="group2", hover_name="compound",
                 title="Vina score vs molecule size", labels={"heavy_atoms": "Heavy atoms", "score": "Vina score (kcal/mol)"})
fig.update_traces(marker_size=11)
fig.add_hline(y=-8.19, line_dash="dash", annotation_text="X77 (validation drug)")
fig.write_html(W + "/score_vs_size.html")
fig.show()

                              count  mean
group2                                   
HIV protease inhibitor            8 -8.02
covalent / cysteine-reactive      4 -6.39
natural compound                  5 -7.52
negative control                  4 -5.31
non-covalent Mpro inhibitor       1 -8.72
other repurposed drug             3 -5.81

Size vs score: rho = -0.82, p = 0.0000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [27]:
import numpy as np

slope, intercept = np.polyfit(df["heavy_atoms"], df["score"], 1)
df["expected_for_size"] = (slope * df["heavy_atoms"] + intercept).round(2)
df["size_adjusted"] = (df["score"] - df["expected_for_size"]).round(2)

RT = 0.001987 * 298.15   # kcal/mol at 25 °C
df["KD_uM"] = (np.exp(df["score"] / RT) * 1e6).round(2)   # micromolar
df.to_csv(W + "/docking_results_full.csv", index=False)

cols = ["compound", "group2", "heavy_atoms", "score", "size_adjusted", "KD_uM"]
print(df.sort_values("size_adjusted")[cols].to_string(index=False))
print(f"\nTrend: each extra heavy atom adds about {slope:.2f} kcal/mol of 'free' score")

          compound                       group2  heavy_atoms  score  size_adjusted  KD_uM
          luteolin             natural compound           21 -7.807          -1.46   1.89
         myricetin             natural compound           23 -7.795          -1.28   1.93
       ensitrelvir  non-covalent Mpro inhibitor           37 -8.722          -1.06   0.40
         quercetin             natural compound           22 -7.424          -0.99   3.61
         baicalein             natural compound           20 -7.169          -0.90   5.55
        nelfinavir       HIV protease inhibitor           40 -8.580          -0.67   0.51
          curcumin             natural compound           27 -7.383          -0.54   3.87
        boceprevir covalent / cysteine-reactive           37 -7.915          -0.25   1.58
         ibuprofen             negative control           15 -6.107          -0.25  33.35
      nirmatrelvir covalent / cysteine-reactive           35 -7.739          -0.24   2.12
         d

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [30]:
import numpy as np, plotly.graph_objects as go
from scipy.integrate import solve_ivp

ka, Rmax = 1e5, 100          # assumed: association rate (1/M/s), max signal (RU)
t_on, t_end = 120, 300       # drug flows 0-120 s, then washout to 300 s
t = np.linspace(0, t_end, 600)

def ideal(KD_uM, C_uM):
    KD, C = KD_uM*1e-6, C_uM*1e-6
    kd = KD*ka
    Req = Rmax*C/(C+KD)
    R_end = Req*(1-np.exp(-(ka*C+kd)*t_on))
    R = np.empty_like(t)
    on = t <= t_on
    R[on] = Req*(1-np.exp(-(ka*C+kd)*t[on]))
    R[~on] = R_end*np.exp(-kd*(t[~on]-t_on))
    return R

def sticky(KD_uM, C_uM, R_ns=25, k_on=0.03, k_off=2e-4):
    ns_end = R_ns*(1-np.exp(-k_on*t_on))
    ns = np.where(t <= t_on, R_ns*(1-np.exp(-k_on*t)), ns_end*np.exp(-k_off*(t-t_on)))
    return ideal(KD_uM, C_uM) + ns

def mtl(KD_uM, C_uM, km=2e6):
    KD, C = KD_uM*1e-6, C_uM*1e-6
    kd = KD*ka
    def rhs(Cc):
        return lambda _, R: [(ka*Cc*(Rmax-R[0]) - kd*R[0]) / (1 + ka*(Rmax-R[0])/km)]
    a = solve_ivp(rhs(C), (0, t_on), [0], t_eval=t[t <= t_on])
    b = solve_ivp(rhs(0), (t_on, t_end), [a.y[0][-1]], t_eval=t[t > t_on])
    return np.concatenate([a.y[0], b.y[0]])

kd_of = df.set_index("compound")["KD_uM"]

# Figure 1: three compounds, same concentration
fig1 = go.Figure()
for name in ["ensitrelvir", "luteolin", "aspirin"]:
    fig1.add_trace(go.Scatter(x=t, y=ideal(kd_of[name], 10),
                              name=f"{name} (K_D {kd_of[name]:.2f} µM)"))
fig1.update_layout(title="Simulated sensorgrams at 10 µM (ideal 1:1 binding)",
                   xaxis_title="Time (s)", yaxis_title="Response (RU)")
fig1.write_html(W + "/sensorgram_compounds.html"); fig1.show()

# Figure 2: one compound, ideal vs artifacts
n, c = "ensitrelvir", 1
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t, y=ideal(kd_of[n], c), name="A: ideal 1:1"))
fig2.add_trace(go.Scatter(x=t, y=sticky(kd_of[n], c), name="B: sticky (non-specific)"))
fig2.add_trace(go.Scatter(x=t, y=mtl(kd_of[n], c), name="C: mass-transport limited"))
fig2.update_layout(title=f"{n} at {c} µM: ideal vs artifacts",
                   xaxis_title="Time (s)", yaxis_title="Response (RU)")
fig2.write_html(W + "/sensorgram_artifacts.html"); fig2.show()